# Fetch and cache popular games' reviews

Run all cells to read `data/popular_game_ids.json`, skip games for which `has_fetched_reviews` returns true (more than 50 CSV review records), and fetch one page of up to 100 reviews for each remaining game. Each response is saved immediately to `data/reviews/<appid>.csv` using the existing client and CSV helpers.

There is a five-second pause between request completion and the next request. Failed games are reported and the pipeline continues; interrupting the notebook preserves CSVs already saved. Rerunning checks the cache again. Games with 50 or fewer saved reviews remain eligible; the existing save helper appends rows, including duplicates on repeated fetches.


In [4]:
import json
import sys
import time
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src").is_dir() and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook from the project root or notebooks folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.csv_io import has_fetched_reviews, save_reviews
from src.steam_reviews_client import get_game_reviews

IDS_FILE = PROJECT_ROOT / "data" / "popular_game_ids.json"
REQUEST_DELAY = 2


In [5]:
app_ids = json.loads(IDS_FILE.read_text(encoding="utf-8"))
if not isinstance(app_ids, list) or any(
    type(app_id) is not int or not 1 <= app_id <= 2**32 - 1
    for app_id in app_ids
):
    raise ValueError("Expected a JSON list of positive integer Steam app IDs")
app_ids = list(dict.fromkeys(app_ids))
print(f"Loaded {len(app_ids)} unique game IDs")


Loaded 1000 unique game IDs


In [6]:
fetched = 0
skipped = 0
failures = []
last_request_finished = None

for index, app_id in enumerate(app_ids, start=1):
    if has_fetched_reviews(app_id):
        skipped += 1
        print(f"[{index}/{len(app_ids)}] Skipping {app_id}: reviews already fetched", flush=True)
        continue

    # Cache checks do not incur a delay; space actual requests five seconds apart.
    if last_request_finished is not None:
        remaining = REQUEST_DELAY - (time.monotonic() - last_request_finished)
        if remaining > 0:
            time.sleep(remaining)

    print(f"[{index}/{len(app_ids)}] Fetching reviews for {app_id}...", flush=True)
    try:
        try:
            response = get_game_reviews(appid=app_id)
        finally:
            last_request_finished = time.monotonic()
        save_reviews(app_id, response)
    except (RuntimeError, OSError, ValueError) as exc:
        failures.append({"appid": app_id, "error": str(exc)})
        print(f"Failed {app_id}: {exc}", flush=True)
        continue

    fetched += 1
    count = len(response.get("response", {}).get("reviews", []))
    print(f"Saved {count} reviews for {app_id}", flush=True)

print(f"Finished: {fetched} fetched, {skipped} skipped, {len(failures)} failed")
failures


[1/1000] Fetching reviews for 730...
Saved 100 reviews for 730
[2/1000] Fetching reviews for 1172470...
Saved 100 reviews for 1172470
[3/1000] Fetching reviews for 578080...
Saved 100 reviews for 578080
[4/1000] Fetching reviews for 1623730...
Saved 100 reviews for 1623730
[5/1000] Fetching reviews for 440...
Saved 100 reviews for 440
[6/1000] Fetching reviews for 1938090...
Saved 100 reviews for 1938090
[7/1000] Fetching reviews for 1063730...
Saved 100 reviews for 1063730
[8/1000] Fetching reviews for 2358720...
Saved 100 reviews for 2358720
[9/1000] Fetching reviews for 271590...
Saved 100 reviews for 271590
[10/1000] Fetching reviews for 550...
Saved 100 reviews for 550
[11/1000] Fetching reviews for 304930...
Saved 100 reviews for 304930
[12/1000] Fetching reviews for 1599340...
Saved 100 reviews for 1599340
[13/1000] Fetching reviews for 236390...
Saved 100 reviews for 236390
[14/1000] Fetching reviews for 2246340...
Saved 100 reviews for 2246340
[15/1000] Fetching reviews for 55

[{'appid': 699130,
  'error': 'Could not reach Steam: <urlopen error _ssl.c:983: The handshake operation timed out>'}]